In [ ]:
"""
So Sánh Các Chiến Lược Điền Khuyết Dữ Liệu
"""

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
# IterativeImputer hiện vẫn nằm trong module experimental của sklearn
from sklearn.experimental import enable_iterative_imputer  # Bắt buộc phải import dòng này trước
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# 1. Tạo tập dữ liệu mô phỏng có tương quan thực tế
# Logic: Lương (Salary) phụ thuộc tuyến tính vào Số năm kinh nghiệm (Experience)
np.random.seed(42)
n = 10

experience = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
# Công thức gốc: Salary = Experience * 1500 + 5000 + Noise
salary = experience * 1500 + 5000 + np.random.normal(0, 500, size=n)

# Cố tình tạo khuyết thiếu theo cơ chế MAR
experience[3] = np.nan  # Khuyết ở dòng index 3
salary[3] = 11000.0     # Dòng này có salary nhưng thiếu exp
salary[7] = np.nan      # Dòng này có exp=8 nhưng thiếu salary

df_sample = pd.DataFrame({
    "Experience": experience,
    "Salary": salary
})

print("=== DỮ LIỆU GỐC CHỨA KHUYẾT THIẾU ===")
print(df_sample)

# 2. Điền khuyết cơ bản: Mean vs Median 
mean_imputer = SimpleImputer(strategy="mean")
df_mean = pd.DataFrame(mean_imputer.fit_transform(df_sample), columns=df_sample.columns)

median_imputer = SimpleImputer(strategy="median")
df_median = pd.DataFrame(median_imputer.fit_transform(df_sample), columns=df_sample.columns)

# 3. Điền khuyết nâng cao: KNNImputer (K=2)
knn_imputer = KNNImputer(n_neighbors=2, weights="distance")
df_knn = pd.DataFrame(knn_imputer.fit_transform(df_sample), columns=df_sample.columns)

# 4. Điền khuyết nâng cao: IterativeImputer (MICE với Bayesian Ridge)
mice_imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=42)
df_mice = pd.DataFrame(mice_imputer.fit_transform(df_sample), columns=df_sample.columns)

# 5. Tạo cờ báo khuyết (Missing Indicator)
indicator = MissingIndicator(features="missing-only")
missing_flags = indicator.fit_transform(df_sample)
flag_cols = [f"{col}_is_missing" for col in df_sample.columns[indicator.features_]]
df_flags = pd.DataFrame(missing_flags, columns=flag_cols)

# 6. Tổng hợp so sánh kết quả tại các dòng bị khuyết (Dòng 3 và Dòng 7)
comparison = pd.DataFrame({
    "Original_Exp": df_sample["Experience"],
    "Mean_Exp": df_mean["Experience"].round(1),
    "KNN_Exp": df_knn["Experience"].round(1),
    "MICE_Exp": df_mice["Experience"].round(1),
    "Original_Sal": df_sample["Salary"],
    "Mean_Sal": df_mean["Salary"].round(1),
    "KNN_Sal": df_knn["Salary"].round(1),
    "MICE_Sal": df_mice["Salary"].round(1),
})

print("\n=== ĐỐI CHIẾU KẾT QUẢ ĐIỀN KHUYẾT TẠI CÁC DÒNG 3 VÀ 7 ===")
print(comparison.iloc[[3, 7]])

print("\n=== MA TRẬN CỜ BÁO KHUYẾT (MISSING INDICATORS) ===")
print(df_flags.iloc[[3, 7]])

=== DỮ LIỆU GỐC CHỨA KHUYẾT THIẾU ===
   Experience        Salary
0         1.0   6748.357077
1         2.0   7930.867849
2         3.0   9823.844269
3         NaN  11000.000000
4         5.0  12382.923313
5         6.0  13882.931522
6         7.0  16289.606408
7         8.0           NaN
8         9.0  18265.262807
9        10.0  20271.280022

=== ĐỐI CHIẾU KẾT QUẢ ĐIỀN KHUYẾT TẠI CÁC DÒNG 3 VÀ 7 ===
   Original_Exp  Mean_Exp  KNN_Exp  MICE_Exp  Original_Sal  Mean_Sal  KNN_Sal  \
3           NaN       5.7      3.9       3.9       11000.0   11000.0  11000.0   
7           8.0       8.0      8.0       8.0           NaN   12955.0  17277.4   

   MICE_Sal  
3   11000.0  
7   17134.9  

=== MA TRẬN CỜ BÁO KHUYẾT (MISSING INDICATORS) ===
   Experience_is_missing  Salary_is_missing
3                   True              False
7                  False               True


In [ ]:
"""
Trắc Lượng & Xử Lý Ngoại Lai: IQR vs Z-Score vs Modified Z-Score
 DUT AI Board - Data Science & Machine Learning Course
"""

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# 1. Khởi tạo dữ liệu thu nhập mô phỏng (gồm phân phối chuẩn + ngoại lai cực đoan)
np.random.seed(42)
normal_income = np.random.normal(loc=1500, scale=300, size=100) # Lương thông thường ($1500 +- 300)
# Cố tình đưa vào các ngoại lai lỗi và ngoại lai tự nhiên
outliers_error = np.array([-500, 99999])  # Lỗi số âm và gõ thừa số 9
outliers_legit = np.array([5500, 6200])   # Lương giám đốc điều hành

income_data = np.concatenate([normal_income, outliers_error, outliers_legit])
df = pd.DataFrame({"Income": income_data})

print(f"Tổng số mẫu: {len(df)}")
print(f"Giá trị Min: {df['Income'].min():.2f} | Max: {df['Income'].max():.2f}")

# 2. Phương pháp 1: IQR Rule (Tukey)
Q1 = df["Income"].quantile(0.25)
Q3 = df["Income"].quantile(0.75)
IQR = Q3 - Q1
iqr_lower = Q1 - 1.5 * IQR
iqr_upper = Q3 + 1.5 * IQR
iqr_outliers = df[(df["Income"] < iqr_lower) | (df["Income"] > iqr_upper)]

# 3. Phương pháp 2: Z-Score truyền thống
mean = df["Income"].mean()
std = df["Income"].std()
z_scores = (df["Income"] - mean) / std
z_outliers = df[np.abs(z_scores) > 3]

# 4. Phương pháp 3: Modified Z-Score (MAD)
median = df["Income"].median()
mad = np.median(np.abs(df["Income"] - median))
modified_z_scores = 0.6745 * (df["Income"] - median) / mad
mod_z_outliers = df[np.abs(modified_z_scores) > 3.5]

print("\n=== KẾT QUẢ SO SÁNH PHÁT HIỆN NGOẠI LAI ===")
print(f"IQR Rule (Ngưỡng [{iqr_lower:.1f}, {iqr_upper:.1f}]): Tìm thấy {len(iqr_outliers)} ngoại lai")
print(f"Z-Score truyền thống (|Z| > 3): Tìm thấy {len(z_outliers)} ngoại lai (Bị Masking!)")
print(f"Modified Z-Score (|M| > 3.5): Tìm thấy {len(mod_z_outliers)} ngoại lai (Kháng Masking thành công)")

print("\nChi tiết các ngoại lai phát hiện bởi Modified Z-Score:")
print(mod_z_outliers)

# 5. Thực thi xử lý ngoại lai: Winsorization (Clipping)
# Ép toàn bộ giá trị về ngưỡng hợp lệ của IQR
df["Income_Cleaned_Clip"] = df["Income"].clip(lower=iqr_lower, upper=iqr_upper)

# Biến đổi Log Transformation (áp dụng trên phần giá trị dương)
df_positive = df[df["Income"] > 0].copy()
df_positive["Income_Log"] = np.log1p(df_positive["Income"])

print("\nThống kê mô tả sau khi Clipping:")
print(df["Income_Cleaned_Clip"].describe())

Tổng số mẫu: 104
Giá trị Min: -500.00 | Max: 99999.00

=== KẾT QUẢ SO SÁNH PHÁT HIỆN NGOẠI LAI ===
IQR Rule (Ngưỡng [815.1, 2160.9]): Tìm thấy 5 ngoại lai
Z-Score truyền thống (|Z| > 3): Tìm thấy 1 ngoại lai (Bị Masking!)
Modified Z-Score (|M| > 3.5): Tìm thấy 4 ngoại lai (Kháng Masking thành công)

Chi tiết các ngoại lai phát hiện bởi Modified Z-Score:
      Income
100   -500.0
101  99999.0
102   5500.0
103   6200.0

Thống kê mô tả sau khi Clipping:
count     104.000000
mean     1483.492206
std       296.389431
min       815.054984
25%      1319.728299
50%      1471.851239
75%      1656.177176
max      2160.850491
Name: Income_Cleaned_Clip, dtype: float64


In [1]:
"""
Chuẩn Hóa Dữ Liệu Bất Nhất Quán: Duplicates, Strings, Currency & Mapping
 DUT AI Board - Data Science & Machine Learning Course
"""

import re
import pandas as pd
import numpy as np

# 1. Tạo tập dữ liệu bẩn mô phỏng
dirty_records = [
    {"ID": "C01", "Name": "  nguyễn  văn an ", "Salary": "$1,200.50", "Gender": "male", "City": "Đà Nẵng", "Join_Date": "15/01/2023"},
    {"ID": "C02", "Name": "Lê Thị Bình\n", "Salary": "25000000 VND", "Gender": "Nữ", "City": "Da Nang", "Join_Date": "2023-05-20"},
    {"ID": "C03", "Name": "TRẦN VĂN CƯỜNG", "Salary": "$3,450.00", "Gender": "M", "City": "Hà Nội", "Join_Date": "01/12/2022"},
    {"ID": "C01", "Name": "nguyễn văn an", "Salary": "$1,200.50", "Gender": "Nam", "City": "Đà Nẵng", "Join_Date": "15/01/2023"}, # Trùng ID C01
    {"ID": "C04", "Name": "Phạm Hồng Đào", "Salary": "N/A", "Gender": "female", "City": "Tp. Hồ Chí Minh", "Join_Date": "31/02/2023"}, # Ngày vô lý (31/2)
]

df = pd.DataFrame(dirty_records)
print("=== DỮ LIỆU BAN ĐẦU CHỨA NHIỀU BẤT NHẤT QUÁN ===")
print(df)

# 2. Xử lý trùng lặp theo khóa chính ID
print(f"\nSố dòng trước khi khử trùng: {len(df)}")
df = df.drop_duplicates(subset=["ID"], keep="first").reset_index(drop=True)
print(f"Số dòng sau khi khử trùng theo ID: {len(df)}")

# 3. Chuẩn hóa chuỗi họ tên
df["Name_Clean"] = (
    df["Name"].astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

# 4. Làm sạch và trích xuất cột lương sang float (USD)
def parse_salary_to_usd(val: str) -> float:
    """Tự động nhận diện VND quy đổi sang USD (1 USD ~ 25,000 VND) và trích xuất số."""
    if pd.isna(val) or val == "N/A":
        return np.nan
    val_str = str(val).upper().replace(",", "")
    # Tìm tất cả chữ số và dấu chấm thập phân
    numbers = re.findall(r"\d+\.?\d*", val_str)
    if not numbers:
        return np.nan
    amount = float(numbers[0])
    if "VND" in val_str:
        amount = amount / 25000.0  # Quy đổi về USD để đồng nhất đơn vị (Uniformity)
    return round(amount, 2)

df["Salary_USD"] = df["Salary"].apply(parse_salary_to_usd)

# 5. Ánh xạ chuẩn hóa cột Giới tính
gender_dict = {
    "male": "Nam", "m": "Nam", "nam": "Nam",
    "female": "Nữ", "f": "Nữ", "nữ": "Nữ"
}
df["Gender_Clean"] = df["Gender"].str.lower().str.strip().map(gender_dict).fillna("Chưa rõ")

# 6. Chuẩn hóa ngày tháng
# Sử dụng mixed format để tự động thích ứng với nhiều định dạng
df["Join_Date_Clean"] = pd.to_datetime(df["Join_Date"], format="mixed", errors="coerce")

print("\n=== DỮ LIỆU SAU KHI ĐƯỢC CHUẨN HÓA TOÀN DIỆN ===")
print(df[["ID", "Name_Clean", "Salary_USD", "Gender_Clean", "Join_Date_Clean"]])
print("\nKiểu dữ liệu sau chuẩn hóa:")
print(df[["Salary_USD", "Join_Date_Clean"]].dtypes)

=== DỮ LIỆU BAN ĐẦU CHỨA NHIỀU BẤT NHẤT QUÁN ===
    ID               Name        Salary  Gender             City   Join_Date
0  C01    nguyễn  văn an      $1,200.50    male          Đà Nẵng  15/01/2023
1  C02      Lê Thị Bình\n  25000000 VND      Nữ          Da Nang  2023-05-20
2  C03     TRẦN VĂN CƯỜNG     $3,450.00       M           Hà Nội  01/12/2022
3  C01      nguyễn văn an     $1,200.50     Nam          Đà Nẵng  15/01/2023
4  C04      Phạm Hồng Đào           N/A  female  Tp. Hồ Chí Minh  31/02/2023

Số dòng trước khi khử trùng: 5
Số dòng sau khi khử trùng theo ID: 4

=== DỮ LIỆU SAU KHI ĐƯỢC CHUẨN HÓA TOÀN DIỆN ===
    ID      Name_Clean  Salary_USD Gender_Clean Join_Date_Clean
0  C01   Nguyễn Văn An      1200.5          Nam      2023-01-15
1  C02     Lê Thị Bình      1000.0           Nữ      2023-05-20
2  C03  Trần Văn Cường      3450.0          Nam      2022-01-12
3  C04   Phạm Hồng Đào         NaN           Nữ             NaT

Kiểu dữ liệu sau chuẩn hóa:
Salary_USD           

Tự Xây Dựng Custom Transformer Chuẩn Scikit-Learn
Mặc dù Scikit-Learn cung cấp sẵn nhiều transformer mạnh mẽ (SimpleImputer, StandardScaler), thư viện này không có sẵn transformer để tự động cắt tỉa ngoại lai (Outlier Clipping theo IQR).

Để tích hợp liền mạch vào sklearn.pipeline.Pipeline, ta có thể tự viết một lớp Python kế thừa hai lớp cơ sở:

BaseEstimator: Cung cấp miễn phí các phương thức quản lý tham số như .get_params() và .set_params() phục vụ việc Hyperparameter Tuning qua GridSearchCV.
TransformerMixin: Tự động sinh ra phương thức .fit_transform() từ hai hàm con .fit() và .transform().

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class OutlierClipperTransformer(BaseEstimator, TransformerMixin):
    """
    Custom Transformer tự động tính ngưỡng IQR trên tập Train và cắt tỉa (Clip)
    các giá trị ngoại lai trên cả tập Train lẫn Test.
    """
    def __init__(self, factor: float = 1.5):
        self.factor = factor
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}

    def fit(self, X, y=None):
        """
        Học các ngưỡng cận dưới và cận trên từ tập Train.
        X có thể là DataFrame hoặc numpy array.
        """
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        self.lower_bounds_ = {}
        self.upper_bounds_ = {}

        for col in X.columns:
            # Bỏ qua các giá trị NaN khi tính phân vị
            valid_series = X[col].dropna()
            if len(valid_series) > 0:
                q1 = valid_series.quantile(0.25)
                q3 = valid_series.quantile(0.75)
                iqr = q3 - q1
                self.lower_bounds_[col] = q1 - self.factor * iqr
                self.upper_bounds_[col] = q3 + self.factor * iqr
            else:
                self.lower_bounds_[col] = -np.inf
                self.upper_bounds_[col] = np.inf

        return self  # Luôn trả về self theo chuẩn Scikit-Learn

    def transform(self, X):
        """
        Áp dụng ngưỡng đã học để cắt tỉa (clip). Không tính toán lại phân vị!
        """
        X_out = X.copy()
        is_numpy = isinstance(X_out, np.ndarray)
        if is_numpy:
            X_out = pd.DataFrame(X_out)

        for col in X_out.columns:
            if col in self.lower_bounds_:
                X_out[col] = X_out[col].clip(
                    lower=self.lower_bounds_[col],
                    upper=self.upper_bounds_[col]
                )

        return X_out.to_numpy() if is_numpy else X_out

Kiến Trúc ColumnTransformer Đa Tuyến (Multi-branch Pipeline)
Trong một tập dữ liệu thực tế, các thuộc tính luôn có kiểu dữ liệu đa dạng:

Biến số (Numerical Features): Cần điền khuyết bằng Median 
→
→ Cắt tỉa ngoại lai bằng OutlierClipperTransformer 
→
→ Chuẩn hóa thang đo bằng StandardScaler.
Biến phân loại (Categorical Features): Cần điền khuyết bằng nhãn 'Unknown' 
→
→ Mã hóa One-Hot Encoding bằng OneHotEncoder.
Công cụ ColumnTransformer trong Scikit-Learn cho phép ta phân tách luồng dữ liệu thành hai nhánh xử lý song song độc lập, sau đó tự động ghép nối (concatenate) kết quả lại thành một ma trận đầu ra duy nhất:



Pipeline Hoàn Chỉnh Từ Dữ Liệu Thô Đến Mô Hình
Dưới đây là kịch bản hoàn chỉnh minh họa toàn bộ vòng đời: Phân chia tập dữ liệu 
→
→ Lắp ráp Pipeline làm sạch 
→
→ Huấn luyện mô hình 
→
→ Dự báo an toàn trên tập Test:

In [4]:
"""
Xây Dựng Pipeline Làm Sạch Dữ Liệu Tự Động Với Scikit-Learn
 DUT AI Board - Data Science & Machine Learning Course
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin

# 1. Custom Outlier Clipper
class OutlierClipperTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, factor: float = 1.5):
        self.factor = factor
        self.bounds_ = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.bounds_ = {}
        for col in X_df.columns:
            valid_data = X_df[col].dropna()
            if len(valid_data) > 0:
                q1 = np.percentile(valid_data, 25)
                q3 = np.percentile(valid_data, 75)
                iqr = q3 - q1
                self.bounds_[col] = (q1 - self.factor * iqr, q3 + self.factor * iqr)
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        for col, (lower, upper) in self.bounds_.items():
            X_df[col] = X_df[col].clip(lower=lower, upper=upper)
        return X_df.to_numpy()

# 2. Tạo tập dữ liệu mẫu chứa các lỗi thực tế
np.random.seed(42)
n_samples = 300

raw_data = {
    "Age": np.random.choice([22, 35, 50, np.nan, 150], size=n_samples, p=[0.4, 0.35, 0.2, 0.03, 0.02]),
    "Income": np.random.choice([2000, 3500, 5000, np.nan, 99999], size=n_samples, p=[0.35, 0.35, 0.2, 0.05, 0.05]),
    "Credit_Score": np.random.normal(loc=650, scale=80, size=n_samples),
    "City": np.random.choice(["Hà Nội", "Đà Nẵng", "TP.HCM", np.nan], size=n_samples, p=[0.4, 0.3, 0.2, 0.1]),
    "Loan_Approved": np.random.choice([0, 1], size=n_samples, p=[0.6, 0.4]) # Biến mục tiêu
}

df_raw = pd.DataFrame(raw_data)

# 3. Bước sống còn: Tách biệt Features và Target, sau đó Train/Test Split
X = df_raw.drop(columns=["Loan_Approved"])
y = df_raw["Loan_Approved"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Kích thước tập Huấn luyện (Train): {X_train.shape}")
print(f"Kích thước tập Kiểm thử (Test): {X_test.shape}")

# 4. Xác định các danh sách cột
num_cols = ["Age", "Income", "Credit_Score"]
cat_cols = ["City"]

# 5. Xây dựng từng nhánh pipeline chuyên biệt
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier_clipper", OutlierClipperTransformer(factor=1.5)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 6. Hợp nhất qua ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)
    ]
)

# 7. Đóng gói trọn vẹn thành Production Pipeline cùng mô hình học máy
full_model_pipeline = Pipeline([
    ("data_cleaner", preprocessor),
    ("classifier", LogisticRegression())
])

# 8. Huấn luyện (Chỉ học trên X_train và y_train!)
print("\nĐang huấn luyện Pipeline trên tập Huấn luyện...")
full_model_pipeline.fit(X_train, y_train)

# 9. Đánh giá trên tập Kiểm thử (Tuyệt đối không rò rỉ dữ liệu!)
accuracy = full_model_pipeline.score(X_test, y_test)
print(f"Độ chính xác (Accuracy) trên tập Test: {accuracy * 100:.2f}%")

# 10. Dự báo trên một khách hàng mới tinh (New Raw Inference)
new_patient = pd.DataFrame([{
    "Age": 140,              # Bị ngoại lai
    "Income": np.nan,         # Bị khuyết
    "Credit_Score": 720.0,
    "City": "Huế"             # Thành phố chưa từng xuất hiện trong tập Train!
}])

prediction = full_model_pipeline.predict(new_patient)
pred_prob = full_model_pipeline.predict_proba(new_patient)

print("\n=== KẾT QUẢ DỰ BÁO TRÊN DỮ LIỆU INFERENCE MỚI ===")
print(f"Khách hàng mới: {new_patient.to_dict(orient='records')}")
print(f"Kết quả duyệt vay: {'Được duyệt (1)' if prediction[0] == 1 else 'Từ chối (0)'}")
print(f"Xác suất được duyệt: {pred_prob[0][1] * 100:.2f}%")

Kích thước tập Huấn luyện (Train): (240, 4)
Kích thước tập Kiểm thử (Test): (60, 4)

Đang huấn luyện Pipeline trên tập Huấn luyện...
Độ chính xác (Accuracy) trên tập Test: 51.67%

=== KẾT QUẢ DỰ BÁO TRÊN DỮ LIỆU INFERENCE MỚI ===
Khách hàng mới: [{'Age': 140, 'Income': nan, 'Credit_Score': 720.0, 'City': 'Huế'}]
Kết quả duyệt vay: Được duyệt (1)
Xác suất được duyệt: 57.41%
